# Notebook 5: Regularization study (M5)

Two regularization stories on the heart-disease logistic regression:
1. **Lasso path** — sweep $\lambda_1$, watch features die one by one. The features with near-zero target correlation in EDA (BP, FBS over 120, Cholesterol) should be killed first.
2. **L2 (ridge) sweep** — sweep $\lambda_2$, watch the bias-variance tradeoff.

Both use the preprocessed 17-feature representation from notebook 02 so results are comparable to M3/M4.

$$
\min_{\mathbf w, b}\ \frac{1}{N}\sum_{i=1}^N\!\log\!\bigl(1+e^{-(2y_i-1)(\mathbf w^\top \mathbf x_i + b)}\bigr) + \lambda_1 \|\mathbf w\|_1 + \tfrac{\lambda_2}{2}\|\mathbf w\|_2^2
$$

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, time, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, log_loss

SPLITS_DIR = '/content/drive/MyDrive/ECE567_Final/splits'
SEED = 2540
np.random.seed(SEED)

data = np.load(os.path.join(SPLITS_DIR, 'split_preproc.npz'), allow_pickle=True)
X_tr, X_val = data['X_tr'], data['X_val']
y_tr, y_val = data['y_tr'].astype(np.float64), data['y_val'].astype(np.float64)
feature_names = list(data['feature_names'])

with open(os.path.join(SPLITS_DIR, 'results_baselines.json')) as fh:
    LBFGS_AUC = json.load(fh)['sklearn_lbfgs']['val_auc']

print('train:', X_tr.shape, 'val:', X_val.shape, '| features:', len(feature_names))
print(f'L-BFGS unregularized reference val AUC = {LBFGS_AUC:.5f}')

## Part A — Lasso path

sklearn's `LogisticRegression(penalty='l1', solver='saga')` parameterizes regularization as $C = 1/(N\lambda_1)$ (per-sample loss). Sweep `C` from very large (no regularization) down to tiny (everything dies).

In [ ]:
Cs = np.logspace(-5, 3, 25)
lasso_rows = []
coef_path = []  # list of coef vectors

t0 = time.time()
for C in Cs:
    lr = LogisticRegression(penalty='l1', solver='saga', C=C,
                            max_iter=2000, tol=1e-3, n_jobs=-1)
    lr.fit(X_tr, y_tr)
    p_val = lr.predict_proba(X_val)[:, 1]
    w = lr.coef_.ravel()
    n_active = int((np.abs(w) > 1e-8).sum())
    lasso_rows.append({
        'C': C, 'lambda1_per_sample': 1.0 / (X_tr.shape[0] * C),
        'val_auc': roc_auc_score(y_val, p_val),
        'val_logloss': log_loss(y_val, p_val),
        'n_active': n_active,
        'l1_norm': float(np.abs(w).sum()),
    })
    coef_path.append(w)
    print(f'C={C:9.2e}  val_auc={lasso_rows[-1]["val_auc"]:.5f}  '
          f'n_active={n_active:2d}/{len(feature_names)}  '
          f'||w||_1={lasso_rows[-1]["l1_norm"]:.3f}')
print(f'\ntotal sweep time: {time.time()-t0:.1f}s')

lasso_df = pd.DataFrame(lasso_rows)
coef_path = np.array(coef_path)  # shape (n_C, n_features)

### Lasso path figure — coefficient trajectories

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.8))
x_axis = lasso_df['lambda1_per_sample'].values

cmap = plt.cm.tab20(np.linspace(0, 1, coef_path.shape[1]))
for j, name in enumerate(feature_names):
    axes[0].plot(x_axis, coef_path[:, j], color=cmap[j], label=name, lw=1.4)
axes[0].set_xscale('log')
axes[0].set_xlabel(r'$\lambda_1$ (per-sample L1)')
axes[0].set_ylabel('coefficient value')
axes[0].set_title('Lasso coefficient path')
axes[0].axhline(0, color='black', lw=0.5)
axes[0].grid(alpha=0.3)
axes[0].legend(loc='upper left', bbox_to_anchor=(1.0, 1.0), fontsize=7)

ax_auc = axes[1]
ax_act = ax_auc.twinx()
ln1 = ax_auc.plot(x_axis, lasso_df['val_auc'], 'o-', color='steelblue', label='val AUC')
ln2 = ax_act.plot(x_axis, lasso_df['n_active'], 's--', color='crimson', label='# active features')
ax_auc.axhline(LBFGS_AUC, ls=':', color='black', alpha=0.6,
                label=f'L-BFGS unreg = {LBFGS_AUC:.4f}')
ax_auc.set_xscale('log')
ax_auc.set_xlabel(r'$\lambda_1$ (per-sample L1)')
ax_auc.set_ylabel('val AUC', color='steelblue')
ax_act.set_ylabel('# active features', color='crimson')
ax_auc.set_title('Sparsity vs accuracy')
lines = ln1 + ln2
ax_auc.legend(lines, [l.get_label() for l in lines], loc='lower left', fontsize=9)
ax_auc.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(SPLITS_DIR, 'fig_lasso_path.png'), dpi=150, bbox_inches='tight')
plt.show()

### Which features die first?

Order by the $\lambda_1$ at which the coefficient first hits zero (largest → least informative).

In [ ]:
kill_lambda = []
order = np.argsort(lasso_df['lambda1_per_sample'].values)  # low->high lambda
for j, name in enumerate(feature_names):
    first_zero_lambda = np.inf
    for k in order:
        if np.abs(coef_path[k, j]) < 1e-8:
            first_zero_lambda = lasso_df['lambda1_per_sample'].iloc[k]
            break
    kill_lambda.append({'feature': name, 'first_zero_lambda': first_zero_lambda})

kill_df = pd.DataFrame(kill_lambda).sort_values('first_zero_lambda')
print(kill_df.to_string(index=False))

## Part B — L2 (ridge) sweep on logistic regression

In [ ]:
Cs_l2 = np.logspace(-5, 3, 25)
ridge_rows = []
for C in Cs_l2:
    lr = LogisticRegression(penalty='l2', solver='lbfgs', C=C,
                            max_iter=2000, n_jobs=-1)
    lr.fit(X_tr, y_tr)
    p_tr  = lr.predict_proba(X_tr)[:, 1]
    p_val = lr.predict_proba(X_val)[:, 1]
    w = lr.coef_.ravel()
    ridge_rows.append({
        'C': C, 'lambda2_per_sample': 1.0 / (X_tr.shape[0] * C),
        'train_auc'   : roc_auc_score(y_tr,  p_tr),
        'val_auc'     : roc_auc_score(y_val, p_val),
        'val_logloss' : log_loss(y_val, p_val),
        'l2_norm'     : float(np.sqrt((w*w).sum())),
    })
ridge_df = pd.DataFrame(ridge_rows)
print(ridge_df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.5))
x_axis = ridge_df['lambda2_per_sample'].values

axes[0].plot(x_axis, ridge_df['train_auc'], 'o-', label='train AUC', color='steelblue')
axes[0].plot(x_axis, ridge_df['val_auc'],   's-', label='val AUC',   color='crimson')
axes[0].axhline(LBFGS_AUC, ls=':', color='black', alpha=0.6,
                label=f'L-BFGS unreg = {LBFGS_AUC:.4f}')
axes[0].set_xscale('log')
axes[0].set_xlabel(r'$\lambda_2$ (per-sample L2)')
axes[0].set_ylabel('AUC')
axes[0].set_title('Ridge: bias-variance curve')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(x_axis, ridge_df['l2_norm'], 'o-', color='purple')
axes[1].set_xscale('log')
axes[1].set_xlabel(r'$\lambda_2$ (per-sample L2)')
axes[1].set_ylabel(r'$\|w\|_2$')
axes[1].set_title('Coefficient shrinkage')
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(SPLITS_DIR, 'fig_ridge_sweep.png'), dpi=150, bbox_inches='tight')
plt.show()

### Best-val-AUC operating points

In [ ]:
best_lasso = lasso_df.loc[lasso_df['val_auc'].idxmax()]
best_ridge = ridge_df.loc[ridge_df['val_auc'].idxmax()]
print('Lasso best:'); print(best_lasso.to_string()); print()
print('Ridge best:'); print(best_ridge.to_string())

thr = LBFGS_AUC - 0.001
sparse_good = lasso_df[lasso_df['val_auc'] >= thr].sort_values('n_active').head(3)
print('\nMost-sparse Lasso configs still within 0.001 of L-BFGS:')
print(sparse_good[['C', 'lambda1_per_sample', 'n_active', 'val_auc']].to_string(index=False))

### Save

In [ ]:
lasso_df.to_csv(os.path.join(SPLITS_DIR, 'results_lasso_sweep.csv'), index=False)
ridge_df.to_csv(os.path.join(SPLITS_DIR, 'results_ridge_sweep.csv'), index=False)
kill_df.to_csv(os.path.join(SPLITS_DIR, 'results_lasso_feature_deaths.csv'), index=False)
np.save(os.path.join(SPLITS_DIR, 'lasso_coef_path.npy'), coef_path)

out = {
    'lasso_best': best_lasso.to_dict(),
    'ridge_best': best_ridge.to_dict(),
    'lbfgs_ref_auc': LBFGS_AUC,
    'most_sparse_within_001': sparse_good.to_dict(orient='records'),
    'feature_death_order': kill_df.to_dict(orient='records'),
}
with open(os.path.join(SPLITS_DIR, 'results_regularization.json'), 'w') as fh:
    json.dump(out, fh, indent=2, default=float)
print('saved regularization artifacts to splits/')